# TFT training notebook for Colab

Этот ноутбук нужен как рабочий шаблон для Google Colab или любой Python-среды с GPU/CPU.

Что он делает:
- загружает исходные CSV;
- готовит временной ряд для TFT;
- обучает модель на `total_fuel_sales`;
- сохраняет чекпойнт и конфиг;
- экспортирует прогноз в CSV для Power BI.

Если `pytorch_forecasting` окажется тяжёлым для Colab, этот же каркас можно использовать для baseline-модели и потом заменить её на TFT.

## 1. Install dependencies

В Colab эту ячейку лучше запускать первой. Если пакет уже установлен, повторная установка не нужна.

In [ ]:
# Install libraries required for TFT training.
# In Colab, uncomment pip install lines if needed.

# !pip -q install torch pytorch-forecasting lightning pandas numpy scikit-learn

import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('Imports are ready')

## 2. Load data

Здесь есть два варианта:
- загрузить файлы с Google Drive;
- или подцепить CSV, если ты уже их скачал локально.

Ниже шаблон для варианта с Drive. Путь можно заменить под свою структуру.

In [ ]:
# Example path structure in Google Drive:
# /content/drive/MyDrive/TABD/_задание/detailed_data.csv
# /content/drive/MyDrive/TABD/_задание/stations_metadata.csv

DATA_DIR = Path('/content/drive/MyDrive/TABD/_задание')
OUT_DIR = Path('/content/drive/MyDrive/TABD/artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

detailed_path = DATA_DIR / 'detailed_data.csv'
stations_path = DATA_DIR / 'stations_metadata.csv'

df = pd.read_csv(detailed_path)
stations = pd.read_csv(stations_path)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)

print(df.shape)
df.head()

## 3. Basic preprocessing

Важно:
- TFT любит аккуратный индекс времени;
- нужно определить `group_id` для каждой АЗС;
- нужно добавить номер временного шага внутри каждой станции;
- статические признаки нужно держать отдельно от временных.

In [ ]:
# Merge static station attributes.
# This gives the model both time-varying signals and station-level context.

station_static_cols = [
    'station_id', 'road_type', 'direction', 'settlement_size',
    'distance_to_city_km', 'total_pumps', 'shop_area_m2',
    'has_car_wash', 'has_tire_service', 'has_cafe', 'has_hotel', 'has_shop',
    'customer_loyalty_score', 'staff_quality_score',
    'corporate_customer_ratio', 'staff_engagement_score',
    'base_price_AI92', 'base_price_AI95', 'base_price_AI98',
    'base_price_DT_EURO', 'base_price_DT_TANEKO', 'base_price_DT_SUMMER', 'base_price_DT_WINTER',
]

df = df.merge(stations[station_static_cols], on='station_id', how='left', suffixes=('', '_station'))

# Time index within each station.
df['time_idx'] = df.groupby('station_id').cumcount()
df['group_id'] = df['station_id'].astype(str)

# Keep a few derived features that are useful both for TFT and for sanity checks.
df['date'] = df['timestamp'].dt.date
df['day_of_year'] = df['timestamp'].dt.dayofyear
df['weekday'] = df['timestamp'].dt.weekday

df[['station_id', 'timestamp', 'time_idx', 'group_id', 'total_fuel_sales']].head()

## 4. Define train/validation/test split

Для time series нельзя делать случайный split. Делим строго по времени, чтобы не было leakage.

In [ ]:
max_time_idx = df['time_idx'].max()
training_cutoff = int(max_time_idx * 0.70)
validation_cutoff = int(max_time_idx * 0.85)

train_df = df[df['time_idx'] <= training_cutoff].copy()
val_df = df[(df['time_idx'] > training_cutoff) & (df['time_idx'] <= validation_cutoff)].copy()
test_df = df[df['time_idx'] > validation_cutoff].copy()

print('train:', train_df.shape)
print('val:', val_df.shape)
print('test:', test_df.shape)

## 5. Build TFT-ready dataset

Здесь используется `pytorch_forecasting`. Если Colab не даёт его поставить, можно временно оставить только подготовку данных и baseline-модель.

Главная цель этого блока - показать структуру входов для TFT, а не только сам код.

In [ ]:
# The imports are placed in a separate cell so the notebook still reads well
# even when the package is temporarily unavailable.

try:
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import QuantileLoss
    from pytorch_lightning import Trainer
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
    TFT_AVAILABLE = True
except Exception as exc:
    TFT_AVAILABLE = False
    print('pytorch_forecasting is not available yet:', exc)

if TFT_AVAILABLE:
    max_encoder_length = 48
    max_prediction_length = 24

    training = TimeSeriesDataSet(
        train_df,
        time_idx='time_idx',
        target='total_fuel_sales',
        group_ids=['group_id'],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        static_categoricals=['road_type', 'direction', 'settlement_size'],
        static_reals=[
            'distance_to_city_km', 'total_pumps', 'shop_area_m2',
            'customer_loyalty_score', 'staff_quality_score',
            'corporate_customer_ratio', 'staff_engagement_score',
            'base_price_AI92', 'base_price_AI95', 'base_price_AI98',
            'base_price_DT_EURO', 'base_price_DT_TANEKO', 'base_price_DT_SUMMER', 'base_price_DT_WINTER',
        ],
        time_varying_known_categoricals=['season', 'weather_condition', 'ad_channel'],
        time_varying_known_reals=['hour', 'day_of_week', 'week_of_year', 'month', 'quarter', 'temperature', 'precipitation_mm'],
        time_varying_unknown_reals=[
            'total_fuel_sales', 'shop_total_revenue', 'total_traffic',
            'sales_AI92', 'sales_AI95', 'sales_AI98',
            'sales_DT_EURO', 'sales_DT_TANEKO', 'sales_DT_SUMMER', 'sales_DT_WINTER',
        ],
        target_normalizer=GroupNormalizer(groups=['group_id']),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

    batch_size = 64
    train_loader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_loader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)
    print('TFT dataset is ready')
else:
    print('Fallback mode: only data prep is available')

## 6. Train TFT

Если блок выше отработал, здесь запускается обучение. Для первой попытки можно ограничить число эпох, чтобы просто убедиться, что пайплайн работает.

In [ ]:
if TFT_AVAILABLE:
    early_stop_callback = EarlyStopping(monitor='val_loss', patience=3, mode='min')
    lr_logger = LearningRateMonitor()

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=1e-3,
        hidden_size=32,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=16,
        loss=QuantileLoss(),
        log_interval=10,
        reduce_on_plateau_patience=2,
    )

    trainer = Trainer(
        max_epochs=5,
        accelerator='auto',
        devices='auto',
        gradient_clip_val=0.1,
        callbacks=[early_stop_callback, lr_logger],
        enable_checkpointing=True,
        log_every_n_steps=10,
    )

    trainer.fit(tft, train_loader, val_loader)
else:
    print('Skipping training because TFT packages are unavailable')

## 7. Save model and config

Сохраняем не только веса, но и конфиг, чтобы потом можно было загрузить модель и повторно использовать её без пересборки пайплайна.

In [ ]:
if TFT_AVAILABLE:
    ckpt_path = OUT_DIR / 'tft_checkpoint.ckpt'
    tft.save_checkpoint(str(ckpt_path))

    config = {
        'created_at': datetime.utcnow().isoformat(),
        'target': 'total_fuel_sales',
        'max_encoder_length': 48,
        'max_prediction_length': 24,
        'model_type': 'TemporalFusionTransformer',
    }
    with open(OUT_DIR / 'tft_config.json', 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print('Model checkpoint and config saved')
else:
    print('Nothing to save yet')

## 8. Forecast and export for Power BI

В Power BI удобнее всего подгружать уже готовый `forecast.csv` с колонками actual/forecast/bounds.

Если модель обучена, здесь можно сформировать предсказания и выгрузить их в таблицу.

In [ ]:
if TFT_AVAILABLE:
    # Produce predictions for the validation/test windows.
    raw_predictions = tft.predict(val_loader, mode='raw', return_x=True)
    print(type(raw_predictions))

    # Placeholder export structure for Power BI.
    # Replace this with real model outputs after running prediction code.
    forecast_export = test_df[['timestamp', 'station_id']].copy()
    forecast_export['target'] = test_df['total_fuel_sales'].values
    forecast_export['forecast'] = np.nan
    forecast_export['lower_bound'] = np.nan
    forecast_export['upper_bound'] = np.nan
    forecast_export['model_version'] = 'tft_colab_v1'

    forecast_export.to_csv(OUT_DIR / 'forecast.csv', index=False)
    print('forecast.csv exported')
else:
    # Even without TFT we can export a template that Power BI can already read.
    forecast_export = test_df[['timestamp', 'station_id']].copy()
    forecast_export['target'] = test_df['total_fuel_sales'].values
    forecast_export['forecast'] = np.nan
    forecast_export['lower_bound'] = np.nan
    forecast_export['upper_bound'] = np.nan
    forecast_export['model_version'] = 'template_only'
    forecast_export.to_csv(OUT_DIR / 'forecast.csv', index=False)
    print('Template forecast.csv exported')

## 9. Save model metadata for reuse

Этот блок полезен, если потом захочешь быстро загрузить модель и сделать новый прогноз без пересборки всего пайплайна.

In [ ]:
metadata = {
    'model_name': 'TFT',
    'target': 'total_fuel_sales',
    'station_count': int(df['station_id'].nunique()),
    'rows': int(len(df)),
    'train_cutoff': int(training_cutoff),
    'validation_cutoff': int(validation_cutoff),
}
with open(OUT_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata